# Часть 2.3 — Customer Behavior Analysis

3 эксперимента: (1) регрессия `Total_Purchases`; (2) бинарная классификация «активный покупатель» (Total_Purchases ≥ медианы); (3) KMeans-сегментация клиентов.

In [1]:
from pathlib import Path

import kagglehub
import pandas as pd
from _setup import evaluate_models, make_spark
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    ClusteringEvaluator,
    MulticlassClassificationEvaluator,
    RegressionEvaluator,
)
from pyspark.ml.feature import OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.sql import functions as F

SEED = 42
spark = make_spark("hw3-customer")
spark

## Загрузка

In [2]:
csv = next(Path(kagglehub.dataset_download("korpionn/customer-behavior-analysis-dataset")).glob("*.csv"))
df = spark.read.csv(str(csv), header=True, inferSchema=True).drop("User_ID")
df.cache()
print(f"строк: {df.count()}")
df.show(5)

строк: 500
+------+---+---------+-----------+---------------------+------------------+-------------------+---------------+
|Gender|Age| Location|Device_Type|Product_Browsing_Time|Total_Pages_Viewed|Items_Added_to_Cart|Total_Purchases|
+------+---+---------+-----------+---------------------+------------------+-------------------+---------------+
|Female| 23|Ahmedabad|     Mobile|                   60|                30|                  1|              0|
|  Male| 25|  Kolkata|     Tablet|                   30|                38|                  9|              4|
|  Male| 32|Bangalore|    Desktop|                   37|                13|                  5|              0|
|  Male| 35|    Delhi|     Mobile|                    7|                20|                 10|              3|
|  Male| 27|Bangalore|     Tablet|                   35|                20|                  8|              2|
+------+---+---------+-----------+---------------------+------------------+------------------

## Подготовка

In [3]:
categorical = ["Gender", "Location", "Device_Type"]
numeric = ["Age", "Product_Browsing_Time", "Total_Pages_Viewed", "Items_Added_to_Cart"]
median_purchases = df.approxQuantile("Total_Purchases", [0.5], 0.001)[0]
print(f"медиана Total_Purchases: {median_purchases}")

prep = Pipeline(
    stages=[
        *[StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical],
        OneHotEncoder(inputCols=[f"{c}_idx" for c in categorical], outputCols=[f"{c}_oh" for c in categorical]),
        VectorAssembler(inputCols=[*numeric, *[f"{c}_oh" for c in categorical]], outputCol="features_raw"),
        StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False),
    ]
).fit(df)
data = (
    prep.transform(df)
    .withColumn("label", F.col("Total_Purchases").cast("double"))
    .withColumn("is_active", (F.col("Total_Purchases") >= median_purchases).cast("double"))
    .select("features", "label", "is_active", *df.columns)
)
train, test = data.randomSplit([0.8, 0.2], seed=SEED)
train.cache()
test.cache()
print(f"train: {train.count()}, test: {test.count()}")

медиана Total_Purchases: 2.0


train: 426, test: 74


## Эксперимент 1 — регрессия Total_Purchases

In [4]:
reg_evaluators = {
    "RMSE": RegressionEvaluator(labelCol="label", metricName="rmse"),
    "R2": RegressionEvaluator(labelCol="label", metricName="r2"),
}
regressors = {
    "LinearRegression": LinearRegression(featuresCol="features", labelCol="label", maxIter=100),
    "RFRegressor": RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=100, seed=SEED, maxDepth=8),
}
evaluate_models(regressors, train, test, reg_evaluators)

,RMSE,R2
model,,
LinearRegression,1.744006,-0.058908
RFRegressor,1.767182,-0.087239


## Эксперимент 2 — классификация активных покупателей

In [5]:
cls_evaluators = {
    "accuracy": MulticlassClassificationEvaluator(labelCol="is_active", metricName="accuracy"),
    "f1": MulticlassClassificationEvaluator(labelCol="is_active", metricName="f1"),
}
classifiers = {
    "LogReg": LogisticRegression(featuresCol="features", labelCol="is_active", maxIter=100),
    "RandomForest": RandomForestClassifier(
        featuresCol="features", labelCol="is_active", numTrees=100, seed=SEED, maxDepth=6
    ),
}
evaluate_models(classifiers, train, test, cls_evaluators)

,accuracy,f1
model,,
LogReg,0.675676,0.606966
RandomForest,0.743243,0.665814


## Эксперимент 3 — KMeans-сегментация (k=2..5)

In [6]:
sil_eval = ClusteringEvaluator(featuresCol="features", predictionCol="prediction", metricName="silhouette")
rows = []
for k in range(2, 6):
    m = KMeans(k=k, featuresCol="features", seed=SEED, maxIter=20).fit(data)
    rows.append({"k": k, "silhouette": sil_eval.evaluate(m.transform(data)), "WSSSE": m.summary.trainingCost})
exp3 = pd.DataFrame(rows).set_index("k")
exp3

,silhouette,WSSSE
k,,
2,0.210281,7474.323030
3,0.161022,7147.270559
4,0.143283,6873.703117
5,0.240336,6398.094947


In [7]:
best_k = int(exp3["silhouette"].idxmax())
print(f"best k = {best_k}")
best = KMeans(k=best_k, featuresCol="features", seed=SEED, maxIter=20).fit(data)
best.transform(prep.transform(df)).groupBy("prediction").agg(
    F.count("*").alias("n"),
    F.avg("Age").alias("avg_age"),
    F.avg("Product_Browsing_Time").alias("avg_browse"),
    F.avg("Items_Added_to_Cart").alias("avg_cart"),
    F.avg("Total_Purchases").alias("avg_purchases"),
).orderBy("prediction").show()

best k = 5


+----------+---+------------------+------------------+------------------+------------------+
|prediction|  n|           avg_age|        avg_browse|          avg_cart|     avg_purchases|
+----------+---+------------------+------------------+------------------+------------------+
|         0|118|26.152542372881356|31.050847457627118|5.3559322033898304| 2.593220338983051|
|         1|143|26.496503496503497|31.937062937062937|  5.34965034965035|2.4265734265734267|
|         2| 59| 26.11864406779661|30.423728813559322| 5.033898305084746| 2.457627118644068|
|         3| 56|25.553571428571427|27.142857142857142| 4.946428571428571|2.5535714285714284|
|         4|124| 26.54032258064516|30.838709677419356| 4.870967741935484| 2.346774193548387|
+----------+---+------------------+------------------+------------------+------------------+



## Выводы

- **Эксп. 1**: оба регрессора дают **отрицательный R²** (~−0.06) — модели хуже, чем просто предсказывать среднее. Признаки слабо связаны с Total_Purchases на этом датасете в 500 строк.
- **Эксп. 2**: бинарная классификация (≥ медианы) идёт лучше: RF accuracy 0.74 / f1 0.67; LogReg 0.68/0.61. Бинаризация скрывает шум.
- **Эксп. 3**: silhouette 0.14–0.24, чёткой структуры нет; best k=5, но кластеры почти неотличимы по средним метрикам — все имеют avg_purchases ~2.4–2.6.
- Главный вывод: датасет слишком мал и без сильных предикторов — для регрессии не хватает данных, классификация и кластеризация работают только грубо.

In [8]:
spark.stop()